For a node during backward:

```text
1. Get its own gradient
   = upstream gradient(s) coming from downstream
   = if multiple paths, add them together

2. Calculate gradient to parent(s)
   = run the local backward rule
   using:
   - its own/upstream gradient
   - any needed _saved_* values
```

For Step 1:

```text
need: upstream gradient(s)
do NOT need: _saved_* values
```

For Step 2:

```text
need: own/upstream gradient
may need: _saved_* values
need: local backward rule
```

In [33]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.retain_grad()   # y is non-leaf, so keep y.grad explicitly

z = y * 8
z.retain_grad()   # Same idea for z

node = y.grad_fn
print("node:", node)

# next_functions means: "Which backward nodes do I send gradients to next?", it could be multiple.
print("next_functions:", node.next_functions, "\n")

def after_backward(grad_inputs, grad_outputs):
    print(
        "from z to y:", grad_outputs,
        "y to x:", grad_inputs, "\n"
    )
node.register_hook(after_backward)

z.backward()

print("z.grad:", z.grad) # dz/dz = 1
print("y.grad:", y.grad) # dz/dy = 8
print("x.grad:", x.grad)

node: <PowBackward0 object at 0x11c8d05e0>
next_functions: ((<AccumulateGrad object at 0x106dd4fa0>, 0),) 

from z to y: (tensor(8.),) y to x: (tensor(48.),) 

z.grad: tensor(1.)
y.grad: tensor(8.)
x.grad: tensor(48.)


## Let's see `_saved_*` examples

In [37]:
def show_saved(t):
    node = t.grad_fn
    print("\n grad_fn:", node)

    for name in dir(node):
        if name.startswith("_saved"):
            print(name, "=", getattr(node, name))
            
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
show_saved(a * b)

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
show_saved(y)


"""
e^x derivative is still e^x, so saving result is enough
"""
a = torch.tensor(3.0, requires_grad=True)
show_saved(torch.exp(a))


"""
output = self + alpha * other
grad to self
= upstream × 1

grad to other
= upstream × alpha
"""
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
show_saved(torch.add(a, b, alpha = 5))


 grad_fn: <MulBackward0 object at 0x11e2a65c0>
_saved_other = tensor(4., requires_grad=True)
_saved_self = tensor(3., requires_grad=True)

 grad_fn: <PowBackward0 object at 0x11d22d000>
_saved_exponent = 2
_saved_self = tensor(3., requires_grad=True)

 grad_fn: <ExpBackward0 object at 0x106dd53f0>
_saved_result = tensor(20.0855, grad_fn=<ExpBackward0>)

 grad_fn: <AddBackward0 object at 0x106dd53f0>
_saved_alpha = 5
